# 01b · Auditoría de fuga de datos, calidad del corpus y reproducibilidad

### Por qué este notebook va antes de entrenar

Un F1 alto no sirve si el modelo **ya había visto las respuestas**. Eso se llama **fuga de datos** (*data leakage*): información del conjunto de evaluación (*test*) se cuela en el entrenamiento y el resultado sale inflado.

**Analogía:** si en la prueba final aparecen preguntas idénticas a las de la guía de ejercicios, un alumno que memorizó la guía saca buena nota sin haber aprendido. La nota no mide lo que creemos que mide.

Aquí revisamos, con los datos reales del CWLC:

1. **Oraciones repetidas** entre train, dev y test (exactas y casi idénticas).
2. **Anotación inconsistente:** la misma oración etiquetada de formas distintas (ruido de etiquetas).
3. **Menciones ya vistas:** cuántas entidades del test aparecen tal cual en train. No es fuga, pero separa memoria de generalización.
4. **Contaminación por preentrenamiento:** si un modelo pudo leer el texto del test antes de nuestro experimento.
5. **Errores de datos** que pueden romper el entrenamiento: etiquetas BIO inválidas, caracteres extraños, oraciones truncadas.
6. **Reproducibilidad:** huellas digitales (SHA-256) del corpus y prueba de que la misma semilla da el mismo resultado.

Al final se listan las **decisiones** que salen de cada hallazgo. Los resultados se guardan en `results/auditoria_datos.json`.

**Conexión con el curso:** buenas prácticas de evaluación (Unidad 4), sesgos y limitaciones de los modelos fundacionales (Unidad 7) y reporte responsable (TRIPOD-LLM).

In [ ]:
from pathlib import Path
import sys, json
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from datos.cwlc import cargar_split
from datos import auditoria as A
from reproducibilidad import huella_archivos, huella_corpus, verificar_huellas_datos, registrar_huellas_datos, ARCHIVO_HUELLAS

carpeta = RAIZ / "data" / "raw" / "cwlc"
S = {s: cargar_split(carpeta, s) for s in ["train", "dev", "test"]}
{s: len(o) for s, o in S.items()}

## 0. Huellas digitales del corpus

Una **huella digital** es un código SHA-256 calculado a partir del contenido de un archivo. Si cambia una sola letra, la huella cambia por completo. Sirve como "sello de garantía": las huellas esperadas de los 21 archivos del CWLC están guardadas en `configs/huellas_datos.json` (que sí va a git), y cada notebook verifica que los datos en disco sean exactamente esos.

Así, si alguien descarga otra versión del corpus o un archivo se corrompe, lo sabremos antes de entrenar, no después de reportar.

In [ ]:
if ARCHIVO_HUELLAS.exists() and "cwlc" in json.loads(ARCHIVO_HUELLAS.read_text()):
    ok, difs = verificar_huellas_datos(carpeta, "cwlc")
    print("Huellas verificadas ✔" if ok else f"¡ATENCIÓN! archivos distintos a los registrados: {difs}")
else:
    registrar_huellas_datos(carpeta, "cwlc"); print("Huellas registradas por primera vez")

h = huella_archivos(carpeta)
print(f"{h['n_archivos']} archivos · huella combinada {h['combinada'][:16]}…")
print("Huella del contenido por split:", {s: huella_corpus(o)[:12] for s, o in S.items()})

## 1. Oraciones repetidas entre splits

Comparamos los textos **normalizados** (minúsculas, sin tildes, espacios simples), para que "Pcte con HTA" y "pcte con hta" cuenten como la misma oración.

Esperamos encontrar repeticiones: las derivaciones usan mucho texto de plantilla ("se deriva para evaluación y manejo") y descripciones de diagnósticos CIE-10 copiadas del sistema. La pregunta es **cuántas** y **cuántas entidades** afectan.

In [ ]:
dup = A.duplicados_entre_splits(S)
pd.DataFrame({k: {c: v for c, v in d.items() if c != "ejemplos"} for k, d in dup.items()}).T

In [ ]:
print("Ejemplos de oraciones de test que también están en train:")
for t in dup["train∩test"]["ejemplos"]:
    print("  ·", t[:110])

Muchas repeticiones son oraciones muy cortas (una o dos palabras, como un diagnóstico suelto). Repetimos el análisis exigiendo **al menos 5 palabras**, que es donde una repetición sí sugiere texto copiado:

In [ ]:
dup5 = A.duplicados_entre_splits(S, min_tokens=5)
pd.DataFrame({k: {c: v for c, v in d.items() if c != "ejemplos"} for k, d in dup5.items()}).T

## 2. Casi duplicados

Dos oraciones pueden diferir en una palabra y seguir siendo "la misma". Medimos la **similitud de Jaccard** entre los trigramas de palabras (grupos de 3 palabras seguidas): 1 = idénticas, 0 = nada en común. Consideramos casi duplicadas las oraciones del test con similitud ≥ 0,8 con alguna de train, entre oraciones de al menos 6 palabras.

In [ ]:
casi_test = A.casi_duplicados(S["train"], S["test"])
casi_dev = A.casi_duplicados(S["train"], S["dev"])
print({k: v for k, v in casi_test.items() if k != "ejemplos"})
for j, t in casi_test["ejemplos"]:
    print(f"  {j:.2f}  {t}")

## 3. Anotación inconsistente

Si la **misma oración exacta** aparece anotada de dos maneras, el modelo recibe señales contradictorias, y en el test se le puede "castigar" por una respuesta que en train era correcta. Esto pone un techo al F1 alcanzable.

In [ ]:
conf = A.conflictos_de_etiqueta(S)
print({k: v for k, v in conf.items() if k != "ejemplos"})
for t in conf["ejemplos"]:
    print("  ·", t[:110])

## 4. Menciones del test ya vistas en train

Esto **no es fuga**: es normal que "HTA" aparezca en train y en test. Pero si el 90% de las entidades del test ya se vio en train, un F1 alto puede venir de memoria y no de comprensión. Por eso en el notebook 02 reportamos el *recall* por separado para menciones **vistas** y **nuevas**: la nueva es la que mide generalización.

In [ ]:
vistas = pd.DataFrame([A.menciones_vistas(S["train"], S["test"], t) for t in ["Disease", "Medication", "Abbreviation"]])
vistas

## 5. ¿La partición original mezcla derivaciones?

La fuga más grave sería que las oraciones de **una misma derivación** quedaran repartidas entre train y test: el modelo vería el contexto del mismo paciente.

Revisando el código con que PLN@CMM generó estos archivos (repositorio `plncmm/acm_health_msen`), la partición se hizo **en orden** (81% / 9% / 10% de las oraciones), recorriendo las derivaciones una tras otra. Como las oraciones de una derivación quedan contiguas, solo la derivación que cae justo en cada frontera puede quedar partida: **como máximo unas 2 derivaciones**. El riesgo por esta vía es mínimo.

Lo que sí explica las repeticiones de la sección 1 es otra cosa: **texto de plantilla y diagnósticos CIE-10 que se repiten entre pacientes distintos**. Eso es realista (así se escribe en APS), pero conviene medir su efecto.

## 6. Contaminación por preentrenamiento

El modelo `plncmm/roberta-clinical-wl-es` se adaptó con preentrenamiento autosupervisado (lenguaje enmascarado) sobre el "Chilean Waiting List dataset". Su ficha no detalla qué derivaciones usó. Si incluyó las del test del CWLC, el modelo **ya leyó** esas oraciones (sin las etiquetas), lo que podría favorecerlo.

No podemos verificarlo, así que lo **declaramos como limitación**. Algo parecido vale para los LLM (E2): el CWLC es público desde 2022 y pudo estar en sus datos de entrenamiento. Cuando comparemos, una ventaja del modelo chileno se interpretará con esta cautela.

## 7. Calidad de los datos

Errores que no inflan el F1, pero pueden romper o degradar el entrenamiento:

- **Etiquetas BIO inválidas:** un `I-` sin un `B-` antes. Nuestro lector las trata como inicio de entidad.
- **Caracteres raros:** caracteres de control, el símbolo de reemplazo (�) o Unicode no normalizado, que harían que la misma palabra se tokenice distinto.
- **Largo:** oraciones vacías o de 1 palabra, y oraciones que superan el máximo de subpalabras del modelo (lo que pase del límite **se corta y se pierde**, junto con sus entidades).

In [ ]:
bio = A.bio_invalidas(carpeta)
print("I- huérfanas por archivo (solo las distintas de 0):", {k: v for k, v in bio.items() if v})
print(A.caracteres_raros(S))

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("PlanTL-GOB-ES/bsc-bio-ehr-es", add_prefix_space=True)
lar = A.largos(S, tok, 512)
pd.DataFrame(lar).T

## 8. Reproducibilidad: ¿la misma semilla da el mismo resultado?

Entrenamos dos veces un modelo mínimo (200 oraciones, 1 época) con la misma semilla y una vez con otra, y comparamos la **huella de las predicciones**. Con la misma semilla deberían ser idénticas; con otra, distintas.

En CPU esto se cumple bit a bit (lo comprobamos al construir el proyecto). En la GPU del Mac (MPS) algunas operaciones no son deterministas, y pueden aparecer diferencias mínimas: por eso reportamos siempre **3 semillas con media ± DE**. La celda está desactivada por defecto porque descarga y entrena un modelo (unos minutos).

In [ ]:
PROBAR_DETERMINISMO = False

if PROBAR_DETERMINISMO:
    import transformers; transformers.logging.set_verbosity_error()
    from modelos.e1_encoder import Config, entrenar, predecir, dispositivo
    from reproducibilidad import huella_predicciones
    dev_corto = S["dev"][:150]
    for s in [42, 42, 43]:
        cfg = Config("PlanTL-GOB-ES/bsc-bio-ehr-es", "Disease", epocas=1, lr=5e-5, batch=8, semilla=s, max_train=200)
        m, cod, h = entrenar(cfg, S["train"], dev_corto, registro=lambda *a: None)
        p = predecir(m, cod, dev_corto, "Disease", disp=dispositivo())
        print(f"semilla {s}: pérdida {h[0]['perdida']:.10f} · huella predicciones {huella_predicciones(p)[:12]}")

## 9. Decisiones

| Hallazgo | Decisión |
|---|---|
| ~10% de las oraciones del test repetidas en train (≈ 2,7% de las entidades), sobre todo plantillas y diagnósticos CIE-10 | Mantener los *splits* oficiales (para comparar con Báez 2022), **y además** reportar el F1 en el test **sin** esas oraciones |
| Casi duplicados ≈ 3% del test | Se mencionan como limitación; el análisis "sin duplicados" cubre lo principal |
| 42 oraciones con anotación inconsistente (22 cruzan splits) | Ruido de etiquetas: se declara como techo del F1; no se corrige el gold (sería validación humana nueva) |
| 61% de enfermedades y 93% de abreviaturas del test ya vistas en train | Reportar *recall* en menciones **vistas vs nuevas** |
| Partición secuencial: como máximo ~2 derivaciones cruzan fronteras | Riesgo mínimo; se documenta |
| `roberta-clinical-wl-es` pudo ver el texto del test en su preentrenamiento; los LLM pudieron ver el CWLC público | Se declara como limitación al interpretar ventajas de esos modelos |
| 8 etiquetas BIO huérfanas, sin problemas de Unicode | Despreciable; el lector las trata como inicio |
| 9 oraciones de train se truncaban a 256 subpalabras, ninguna a 512 | `max_len = 512` |
| La semilla cambia el resultado | 3 semillas (42, 43, 44), media ± DE; huellas en cada resultado |

El few-shot de E2 tomará ejemplos **solo de train**, y el test se usa una única vez por configuración.

In [ ]:
auditoria = {
    "huellas_contenido": {s: huella_corpus(o) for s, o in S.items()},
    "duplicados_exactos_normalizados": dup, "duplicados_min5tokens": dup5,
    "casi_duplicados_test": casi_test, "casi_duplicados_dev": casi_dev,
    "conflictos": conf, "menciones_vistas": vistas.to_dict("records"),
    "bio_invalidas": bio, "caracteres": A.caracteres_raros(S), "largos": lar,
}
(RAIZ / "results").mkdir(exist_ok=True)
(RAIZ / "results" / "auditoria_datos.json").write_text(json.dumps(auditoria, indent=2, ensure_ascii=False, default=str))
print("Guardado en results/auditoria_datos.json")